# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading and exploring the FAIR² dataset using the `mlcroissant` library, following best practices for referencing data entities by `@id`.

<br>
### Dataset Source
The dataset CROISSANT schema:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata (including record sets and fields)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Loaded: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review available record sets and fields by their `@id`.

For each record set, list its `@id`, name, and `@id`s of its fields/columns.

In [ ]:
# List all record sets in the dataset by @id and list their fields
record_sets = list(dataset.record_sets)
if len(record_sets) == 0:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        print(f"Record Set Name: {rs.name}")
        print(f"Record Set @id: {rs.id}")
        print("Fields (columns) in this record set:")
        for field in rs.fields:
            print(f"  - {field.name} (@id: {field.id})")
        print('-'*60)

## 3. Data Extraction

Load all available data from each record set into Pandas DataFrames for further analysis. 

*All data references use `@id`s for reproducibility.*

In [ ]:
# Get available record set @ids for extraction
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns in record set {record_set_id}: {df.columns.tolist()}")
    if not df.empty:
        display(df.head(3))
    print('-'*50)

# Pick one record set as an example for analysis (use the first available)
if record_set_ids:
    chosen_record_set = record_set_ids[0]
    print(f"Chosen record set for detailed EDA: {chosen_record_set}")
    print(dataframes[chosen_record_set].columns.tolist())
    dataframes[chosen_record_set].head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps: filtering, normalizing, grouping, with references to fields by their `@id`. 

---
*If no numeric fields are present, adjust operations accordingly.*

In [ ]:
# Example: Use one numeric field for processing and grouping

df = dataframes[chosen_record_set]

# List field @ids and try to identify a numeric one
numeric_field_id = None
group_field_id = None
for field in dataset.record_set(chosen_record_set).fields:
    # Try to select the first numeric field for demo
    if hasattr(field, "data_type") and field.data_type:
        if field.data_type.lower() in ["number", "integer", "float"]:
            numeric_field_id = field.id
            break
# Also try to choose a group field (categorical)
if len(df.columns) > 1:
    group_field_id = df.columns[1]  # Pick the second column as default for grouping

if numeric_field_id and numeric_field_id in df.columns:
    # Drop NA for the numeric field
    df_num = df.dropna(subset=[numeric_field_id])
    threshold = df_num[numeric_field_id].mean() if df_num[numeric_field_id].dtype in [int, float] else None
    if threshold is None:
        threshold = 10  # fallback threshold
    filtered_df = df_num[df_num[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    field_norm = f"{numeric_field_id}_normalized"
    filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, field_norm]].head())

    # Group by a categorical field if possible
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric fields identified for EDA. Please adjust the field selection as appropriate for this dataset.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: Visualize histogram of the numeric field if present
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns and df[numeric_field_id].dtype in [int, float, 'float64', 'int64']:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field available for histogram visualization.")

## 6. Conclusion

In this notebook we:

- Loaded the FAIR² dataset and meta-information with `mlcroissant`
- Explored record sets and their field `@id`s for reliable data referencing
- Extracted all data into Pandas DataFrames
- Demonstrated field-based filtering, normalization, and grouping referencing `@id` fields 
- Visualized data (where applicable)

Use this workflow as a foundation for more detailed clinical and molecular analysis. For further use, adapt the field and record set `@id`s based on your analysis needs.